# The arithmetic is the harness's job, and then the critic has one

`07` ended on a verdict and a list. The verdict: a critic that reads the policy's hidden
state cannot work on this task, because the reward is decided by the completion rather than
by the prompt, and a value function that could predict it from the prompt would already have
solved the problem the run exists to solve. The list was three things that would change that
answer, none of them a hyperparameter — a dense or intermediate reward, a prompt-difficulty
signal the model can read, or `gamma < 1`.

Underneath both sat a fact `06` measured and `07`'s closing section sharpened: **Qwen3-1.7B
does not do this task's arithmetic**, and not at the level of the domain formula. Two of the
three percent changes involve no `TCF` at all — a division and a sum, then a percent change —
and it misses those at the same rate as the one that does. Over the 1000 case-evaluations in
`runs/paired/` the count of numbers inside the 0.5 pp tolerance is 844 zeros, 140 ones, 15
twos and one three. `exact_match` is 0.000 at all 27 evaluation points of every run in the
series, because `exact_match` needs the numbers.

This notebook takes the arithmetic away from the model and gives it to the harness. That one
change turns out to supply two items from `07`'s list at once, and it is the reason
everything below happens.

### Three things this had to hit, and how they are judged

| | the goal | the gate |
| --- | --- | --- |
| 1 | the critic is useful **every round** | `value_ev` above `07`'s +0.3 bar — *and* above what a clock scores |
| 2 | the arithmetic is right **every step** | `numeric_acc` = 1.000 at every evaluation point |
| 3 | the policy actually gets better | `exact_match` off the floor it has never left |

The second gate is the one that makes the others askable. The third is the one the series
has never passed.

In [ ]:
import json, sys, collections, statistics as st
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "ppo_ac.py").exists() else Path("experiments/notebooks/smoke_test")
sys.path.insert(0, str(HERE.resolve()))
import ppo_ac

# The check that licenses everything below: the harness's arithmetic has to be
# the same arithmetic the answer key was generated with, to the rounding rule.
for split in ("train", "dev"):
    cases = ppo_ac.load_cases(split)
    bad = [
        (c["id"], k)
        for c in cases
        for k, v in ppo_ac.compute_changes(c["record"]).items()
        if abs(v - c["answer"][k]) > 1e-9
    ]
    print(f"{split}: {len(cases)} cases, {len(bad)} mismatches against the answer key")

print()
print(ppo_ac.build_messages_v3(cases[0]["record"])[1]["content"].split("Step 2")[0][-260:])

## v3 — the readings are already structured, so the harness can just compute

The decisive detail is one nobody in this series wrote down: the twelve readings are **not
free text**. They are already parsed, in `record["t0"]` and `record["t1"]`, and the prompt is
rendered *from* them. So the arithmetic needs no tool-call protocol and no extraction step —
the harness has the numbers, and `compute_changes` reproduces all 600 answers in
`data/{train,dev}.jsonl` exactly. That check is what licenses the whole notebook: if the
reimplementation drifted from `task.generate` by even a rounding rule, every number below
would be graded against a different answer key than the one it was computed from.

`build_messages_v3` is v2 with Step 1 replaced — by surgery on v2's own rendered text, not by
re-rendering it, so everything outside the Step 1 block and the closing instruction is
byte-identical and a v2/v3 comparison has one variable. `v3_prompts()` rebinds the one name
`eval.generate_hf` closes over, so the generation and scoring path stays literally theirs.

### The frozen policy, greedy, full dev split, no training at all

| | v2 (model computes) | **v3 (harness computes)** |
| --- | --- | --- |
| `numeric_acc` | 0.035 | **1.000** |
| `exact_match` | **0.000** | **0.090** |
| `schema_ok` | 0.965 | 1.000 |
| `flags_acc` | 0.323 | **0.590** |
| `cause_given_flags` | **0.000** | **1.000** |
| `cause_acc` | 0.235 | 0.295 |
| `action_acc` | 0.175 | 0.210 |

Three of these deserve to be read slowly.

**`numeric_acc` 1.000.** The model copies twelve digits into the right slots without error.
The capability that was missing was the arithmetic, and only the arithmetic.

**`exact_match` 0.090.** Eighteen completely correct answers, from the *frozen* model, before
a single gradient step. The series had produced zero across sixteen runs.

**`cause_given_flags` 0.000 → 1.000.** This is the one that reframes `07`. Given three correct
flags, the frozen policy under v2 never once read the right row off the decision table; under
v3 it always does. The table lookup was never the missing skill — the arithmetic cascade
upstream of it was poisoning every input it ever got.

And `flags_acc` 0.590 is now the binding step. Turning a *given* decimal into `down`/`up`/
`flat` against a stated threshold is not arithmetic and is not knowledge; it is the kind of
partially-right behaviour that RL can sharpen, and it is what the rest of this notebook is
about. The README's own reason for rejecting Qwen2.5-0.5B was that its `cause_acc` "sat
*exactly* at chance, leaving RL nothing partial to sharpen". Under v2 the numeric half of
this task was in that position. Under v3 it is not.

In [ ]:
def paired(name):
    return json.load(open(HERE / "runs" / "paired" / f"{name}.json"))["overall"]

KEYS = ("numeric_acc", "exact_match", "schema_ok", "flags_acc",
        "cause_given_flags", "cause_acc", "action_acc")
v2, v3 = paired("base"), paired("base_v3_main")
print("the frozen policy, greedy, 200 dev cases, no training")
print(f"  {'':20}{'v2':>10}{'v3':>10}")
for k in KEYS:
    a, b = v2[k], v3[k]
    fmt = lambda x: "  none  " if x is None else f"{x:.3f}"
    print(f"  {k:20}{fmt(a):>10}{fmt(b):>10}")
print("  v2 predicted causes:", v2["predicted_cause_hist"])
print("  v3 predicted causes:", v3["predicted_cause_hist"])

## Why v3 is a different problem for the critic, and why that was not enough

`07` §4 measured what makes a prompt easy under v2, and the answer was the answer itself:
`root_cause` one-hot explains **+0.902** of the prompt-mean reward leave-one-out, while the
variables the task was *designed* around — `tier`, `margin_pp`, `severe` — explain **−0.081**.
A value head could not predict the reward without first solving the task.

Under v3 the three percent changes are printed in the prompt, and the thresholds are constant.
So difficulty becomes *how close those printed numbers sit to their thresholds* — a computable
function of text the model is looking at. Measured on the frozen v3 run, three hand-made
margin features give **LOO R² = +0.140** against per-case reward, with `corr = +0.34` for the
smallest margin alone. That is not large, but it is the right sign, and it is the first time
in this series that difficulty has been readable from the prompt at all.

### It still was not enough: the run that was stopped

The first v3 training run kept the terminal reward and `lam = 1.0`, changing only the prompt
and adding `critic_window=25`. It was stopped at step 25:

| | |
| --- | --- |
| `value_ev` median | **+0.065** — against `07`'s +0.3 bar |
| `value_ev_fit` (in sample) | +0.137 |

`07`'s own guidance names this case: if the head cannot fit the window it was just trained on,
the window is not the constraint. And the reason is the structural one `07` closes with and
which v3 does not touch — with `gamma = lam = 1` and one terminal reward the return is
**constant along a sequence**, so a causal `V(s_t)` can only distinguish between prompts, and
within a batch of eight the return spread is about 5% of the mean. The prompt got easier to
read; the critic's job stayed impossible. `runs/v3-r1-stopped-at-25/` is the record.

In [ ]:
import numpy as np

# Under v3 the three changes are printed in the prompt and the thresholds are
# constant, so "how close is this case to a boundary" is a function of text the
# model can see. Under v2 the equivalent question needed the answer -- `07` §4.
BOUNDS = {"normalized_flow_change_pct": (-10.0, 10.0),
          "salt_passage_change_pct": (-15.0, 15.0, 50.0),
          "dp_change_pct": (-15.0, 15.0)}
cases = {c["id"]: c for c in ppo_ac.load_cases("dev")}
rows = paired("base_v3_main")["per_case"]

X, Y = [], []
for row in rows:
    ch = ppo_ac.compute_changes(cases[row["id"]]["record"])
    X.append([min(abs(ch[k] - b) for b in bs) for k, bs in BOUNDS.items()])
    Y.append(row["reward"])
X, Y = np.array(X), np.array(Y)

A = np.c_[X, np.ones(len(Y))]
loo = np.array([A[i] @ np.linalg.lstsq(np.delete(A, i, 0), np.delete(Y, i), rcond=None)[0]
                for i in range(len(Y))])
r2 = 1 - ((Y - loo) ** 2).sum() / ((Y - Y.mean()) ** 2).sum()
print(f"three threshold margins -> per-case reward, LOO R^2: {r2:+.3f}")
print(f"  corr(reward, smallest margin):            {np.corrcoef(X.min(1), Y)[0,1]:+.3f}")
print(f"  cases within 5 pp of a threshold:         {(X.min(1) < 5).mean():.1%}")
print(f"  mean reward near a boundary / far from one: "
      f"{Y[X.min(1) < 5].mean():.3f} / {Y[X.min(1) >= 5].mean():.3f}")
print()
print("for comparison, 07 section 4, under v2:")
print("  tier + margin_pp + severe (the designed difficulty)  LOO R^2  -0.081")
print("  root_cause one-hot (the answer itself)               LOO R^2  +0.902")

## Dense per-field reward — paying each field where it is written

This is the first item on `07`'s list, and v3 is what makes it available: the output is a
JSON object whose fields are decided one at a time, in a fixed order, at locatable positions.

`field_credits` splits one completion's reward into per-field pieces — each of the three
numbers and each of the three flags carries a third of its component's weight, since
`reward.py` scores both as a mean of three hits — and `dense_rewards` places each piece on the
token that completed that field, by prefix-decoding to turn a character offset into a token
index. `format` is schema validity, which is not known until the object closes, so it stays
at the last active token along with anything whose field could not be located.

**Row sums are unchanged.** The per-token rewards add up to `score(...).total` to within
float error, so the sequence-level objective the actor optimises is exactly the objective
every earlier run in this series optimised. Only *when* the credit arrives moves.

What that buys is the thing `07` said was missing:

```
return along one sequence, before:  0.71  0.71  0.71  0.71  0.71   <- constant
return along one sequence, after:   0.71  0.71  0.48  0.28  0.25   <- falls as it is written
```

Within-sequence return sd goes from 0 to **0.221**, against a between-sequence `adv_std` of
about 0.035. `V(s_t)` finally has something to track: how much reward is left to be earned.

**It only bites with `lam < 1`.** At `lam = 1` GAE never consults `V` at all, so dense credit
changes the metrics and nothing else. That is not a caveat, it is the ablation — and it is
the one run below that settles whether the critic is doing work or merely scoring well.

In [ ]:
import torch

# What the dense split does to one sequence, with no model involved: build a
# completion that is right about some fields and wrong about others, and read
# off where the credit lands and what the return looks like along the way.
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B", padding_side="left")
tok.pad_token = tok.pad_token or tok.eos_token
case = ppo_ac.load_cases("dev")[0]
a = case["answer"]
text = "dp 0.64+0.93=1.57 -> 1.85\n" + json.dumps({
    "normalized_flow_change_pct": a["normalized_flow_change_pct"],
    "salt_passage_change_pct": a["salt_passage_change_pct"],
    "dp_change_pct": a["dp_change_pct"],
    "flags": {k: a["flags"][k] for k in ("flow", "salt_passage", "dp")},
    "stage": a["stage"], "root_cause": "compaction", "action": a["action"]})

ids = torch.tensor([tok(text, add_special_tokens=False)["input_ids"]])
mask = torch.ones_like(ids, dtype=torch.float32)
W = ppo_ac.WEIGHT_SETS["ABLATE"]
dense = ppo_ac.dense_rewards([text], ids, mask, [case], W, tok)

print(f"row sum {float(dense.sum()):.6f}   score().total {ppo_ac.score(text, a, W).total:.6f}")
placed, terminal = ppo_ac.field_credits(text, a, W)
for field, (offset, credit) in placed.items():
    print(f"  {field:28} char {offset:4}   credit {credit:.4f}")
print(f"  {'format (at the last token)':28}          credit {terminal:.4f}")

ret = dense.flip(-1).cumsum(-1).flip(-1)
n = ids.shape[1] - 1
print("\nreturn along the sequence at 0/25/50/75/100%:",
      [round(float(ret[0, int(p * n)]), 3) for p in (0, .25, .5, .75, 1.0)])
print(f"within-sequence return sd: {float(ret.std()):.3f}   "
      f"(with a terminal reward it is exactly 0, by construction)")

## The three goals, measured

`ppo-qwen3-17b-v3-dense-s0` — v3 prompts, dense per-field credit, `lam = 0.95`, and otherwise
the configuration this series already settled on: `ABLATE`, `entropy_coef = 0.005`, seed 0,
200 steps, `prompts_per_step = 8`, `samples_per_prompt = 1`, `critic_window = 25`,
`value_clip_eps = 0.2`, `recompute_advantages` off.

### Goal 2 — the arithmetic, every step

`numeric_acc` is **1.000 at all nine evaluation points**, as it is in every v3 run including
the frozen one. This is not a result the policy can regress on: the numbers are computed by
`compute_changes` and the model copies them, and it copies them without error.

### Goal 3 — the policy

| step | `exact_match` | `flags_acc` | `cause_acc` | `action_acc` |
| --- | --- | --- | --- | --- |
| 0 | 0.090 | 0.590 | 0.295 | 0.210 |
| 50 | 0.155 | 0.670 | 0.395 | 0.305 |
| 100 | 0.150 | 0.668 | 0.445 | 0.350 |
| 200 | **0.145** | 0.665 | **0.435** | **0.345** |

`exact_match` runs 0.090 → 0.145, peaking at 0.155. **Every run in `runs/` before this one is
0.000 at all 27 of its evaluation points.** `cause_acc` 0.295 → 0.435 and `action_acc`
0.210 → 0.345 are both larger moves than anything `04` or `05` produced, and unlike those they
arrive with the numeric component already saturated, so none of it is format learning and
none of it is the arithmetic component moving.

The honest reading of the shape: most of the gain lands by step 100 and the last hundred steps
are flat-to-noisy, which is what every run in this series does.

### Goal 1 — the critic, and the number that overstates it

| | |
| --- | --- |
| `value_ev` median | **+0.921**, above `07`'s +0.3 bar on **100%** of steps |
| the v2 predecessor's median | −0.009, above +0.3 on 0% of steps |

That comparison is real but it is not the whole story, and the instrument that says so was
added on purpose. **With dense credit the return falls as the answer is written, so a head
that learned nothing except "how far along am I" would already score well.** `value_ev_position`
is that head, fitted for free at every step: the mean return in each of twenty relative-position
bins, on the same batch, knowing nothing about the sequence.

| | |
| --- | --- |
| a clock, position only | **+0.883** |
| critic − clock | **+0.0319** median, critic ahead on 93% of steps |
| first half of the run | +0.0178 |
| second half | **+0.0547** |

So most of that +0.921 is structure I put into the reward, not the critic reading anything.
What the critic adds over counting is small — but it is consistently positive, it grows
threefold across the run, and it is the part that is actually about the sequence. Reporting
+0.921 as "the critic works" would be the same mistake `07` §3 caught in `ValueHead`'s
docstring, where an unsplit fit reported +0.993 and an honest split gave +0.031.

In [ ]:
def evals(run):
    return [json.loads(l) for l in open(HERE / "runs" / run / "eval.jsonl")]

def steps(run):
    return [json.loads(l) for l in open(HERE / "runs" / run / "metrics.jsonl")]

R2, V2 = "ppo-qwen3-17b-v3-dense-s0", "ppo-qwen3-17b-ablate-ent-s0"

print("goal 2 -- the arithmetic, at every evaluation point of the run")
acc = [r["numeric_acc"] for r in evals(R2)]
print(f"  numeric_acc: {acc}  -> min {min(acc):.3f}")
print()
print("goal 3 -- the policy")
print(f"  {'step':>5}{'exact':>8}{'flags':>8}{'cause':>8}{'action':>8}")
for r in evals(R2):
    print(f"  {r['step']:>5}{r['exact_match']:>8.3f}{r['flags_acc']:>8.3f}"
          f"{r['cause_acc']:>8.3f}{r['action_acc']:>8.3f}")
best = max(evals(R2), key=lambda r: r["exact_match"])
print(f"  best exact_match {best['exact_match']:.3f} at step {best['step']};  "
      f"every v2 run in runs/ is 0.000 at all 27 of its evaluation points")

In [ ]:
def critic(run, since=25):
    rows = [r for r in steps(run)[since:]
            if r.get("value_ev") is not None and r.get("value_ev_position") is not None]
    ev = [r["value_ev"] for r in rows]
    clock = [r["value_ev_position"] for r in rows]
    return rows, ev, clock

rows, ev, clock = critic(R2)
v2ev = [r["value_ev"] for r in steps(V2) if r.get("value_ev") is not None]
print("goal 1 -- the critic, over the 175 steps after the window fills")
print(f"  value_ev            median {st.median(ev):+.3f}   above +0.3 on {100*sum(e>0.3 for e in ev)/len(ev):.0f}% of steps")
print(f"  the same run's v2 predecessor            median {st.median(v2ev):+.3f}")
print()
print("  and what a clock would have scored -- the mean return in each of twenty")
print("  relative-position bins, fitted on the same batch, knowing nothing else:")
print(f"  position only       median {st.median(clock):+.3f}")
residual = [a - b for a, b in zip(ev, clock)]
print(f"  critic - clock      median {st.median(residual):+.4f}   "
      f"critic ahead on {100*sum(r>0 for r in residual)/len(residual):.0f}% of steps")
half = len(rows) // 2
for label, sl in (("first half ", slice(0, half)), ("second half", slice(half, None))):
    r = [a - b for a, b in zip(ev[sl], clock[sl])]
    print(f"    {label}      {st.median(r):+.4f}")

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(HERE / "runs" / "v3_dense_three_goals.png")))

## The ablation that settles it: `lam = 1.0`

`value_ev` says the critic is a good estimator. It does not say the critic is *useful*, and
those have come apart before in this series — `07`'s privileged critic reached median +0.799
and moved the task by nothing.

The clean test is available here for free, because of how GAE is written: **at `lam = 1` the
advantage is the Monte-Carlo return minus `V`, and `V` cancels out of the policy gradient
entirely.** So `lam = 1.0` against `lam = 0.95`, same dense reward, same seed, same
everything, is one variable — *is the value function consulted at all* — with the critic still
trained and still measured in both.

| over the 8 evaluation points after step 0 | `lam = 1.0` (V unused) | `lam = 0.95` (V used) | difference |
| --- | --- | --- | --- |
| `exact_match` | 0.1087 | **0.1375** | **+0.029** |
| `cause_acc` | **0.4062** | 0.3981 | −0.008 |
| `flags_acc` | 0.6592 | 0.6587 | −0.000 |
| `action_acc` | **0.3212** | 0.3125 | −0.009 |
| `adv_std` | 0.0655 | **0.0398** | **−39%** |

**The critic does its mechanical job and nothing else.** Advantage spread falls 39% — the
same magnitude `07` measured for the privileged critic under v2, arrived at here without
privileged information. And the task does not move: the four accuracy differences do not
share a sign, the largest is +0.029 on `exact_match`, and the seed band this series has
measured at a fixed configuration is ±0.055. One seed each. The honest statement is **no
measurable task effect**, not that the critic helps and not that it hurts.

That is `07`'s headline confirmed on a task where the critic genuinely works:
**gradient variance is not the binding constraint here.** It was worth re-testing precisely
because `07` could be dismissed — a variance argument made with a critic whose explained
variance was negative proves little. This one is made with a critic at +0.6.

### And a measurement artefact worth recording

The two runs report very different `value_ev` for critics doing the same job:

| | `value_ev` | clock | residual | `adv_std` |
| --- | --- | --- | --- | --- |
| `lam = 1.0` | +0.616 | +0.508 | **+0.0930** | 0.0655 |
| `lam = 0.95` | **+0.921** | +0.883 | +0.0319 | 0.0398 |

`lam = 0.95` looks far better and is worth less. With `lam < 1` the critic is regressed
against the λ-return, which is **partly its own prediction bootstrapped forward**, so it is
scored against a target it helped produce. `lam = 1.0` regresses against the pure Monte-Carlo
return — a target the critic had no hand in — and that is the honest measurement: **+0.616,
+0.093 above a clock, ahead of the clock on 86% of steps.**

So the number to quote for this critic is +0.616, not +0.921. This is the third time in this
series that a value-function number has been inflated by the thing it was measured against —
`ValueHead`'s docstring at +0.993 against an honest +0.031, the run means that hid a +0.419
median behind a −0.314 mean, and now this.

In [ ]:
LAM1 = "ppo-qwen3-17b-v3-dense-lam1-s0"

a = {r["step"]: r for r in evals(LAM1)}
b = {r["step"]: r for r in evals(R2)}
post = [s for s in sorted(a) if s > 0]
print("is the value function consulted?  lam=1.0 vs lam=0.95, one variable")
print(f"  {'metric':14}{'lam=1.0':>10}{'lam=0.95':>11}{'diff':>9}")
for k in ("exact_match", "cause_acc", "flags_acc", "action_acc"):
    x, y = st.mean(a[s][k] for s in post), st.mean(b[s][k] for s in post)
    print(f"  {k:14}{x:>10.4f}{y:>11.4f}{y-x:>+9.4f}")

print()
print("  the critic in each, scored against the target it was actually fitted to")
for label, run in ((f"lam=1.0  (Monte-Carlo target)", LAM1),
                   (f"lam=0.95 (bootstrapped target)", R2)):
    rows, ev, clock = critic(run)
    res = [x - y for x, y in zip(ev, clock)]
    print(f"  {label:32} value_ev {st.median(ev):+.3f}   clock {st.median(clock):+.3f}   "
          f"residual {st.median(res):+.4f}   ahead {100*sum(r>0 for r in res)/len(res):.0f}%   "
          f"adv_std {st.mean(r['adv_std'] for r in rows):.4f}")

## Where this leaves the series

Three goals, and they did not all land the same way.

**The arithmetic: solved, and not by the model.** `numeric_acc` is 1.000 at every evaluation
point of every v3 run, frozen and trained. The capability is still absent from Qwen3-1.7B —
nothing here taught it anything — but the *system* now computes correctly, which is what a
calculator tool would have achieved, minus the tool-call protocol and minus the extraction
step, because `record["t0"]` and `record["t1"]` were already parsed. The honest framing is
that the task was changed, not the model.

**The policy: the first non-zero `exact_match` in the series.** 0.090 frozen, 0.155 at its
best under training, against **0.000 at all 27 evaluation points of every run in `runs/`**
before this notebook. `cause_acc` 0.295 → 0.435 and `action_acc` 0.210 → 0.345, with the
numeric component already saturated so none of it is format learning. The gain is real and it
is small in absolute terms: at 0.145, seven answers in eight are still wrong somewhere.

**The critic: it works, and it does not matter.** Held to the honest measurement — `lam = 1.0`,
Monte-Carlo target, scored against a clock — it reaches +0.616, +0.093 above position-only,
ahead of the clock on 86% of steps, unprivileged. It cuts advantage spread 39%. And the
ablation says the task does not care. That is `07`'s conclusion re-derived on a task where the
critic is no longer broken, which is the version of it worth trusting.

### So what is actually binding now

`flags_acc` moves 0.590 → 0.670 and stops. Everything downstream is gated on it, and the
gating is total: **`cause_given_flags` is 1.000**, so the root cause is not a separate skill
at all — get the three flags right and the answer follows.

And a flag is a decimal compared against a stated threshold, with the decimal supplied. That
is the whole remaining task, and 33% of it is still wrong after 200 steps. What that is not:
it is not arithmetic, not domain knowledge, not the trust region, not the baseline, and not
gradient variance — each of those has now been measured out. The remaining suspects are the
model's handling of the four-way `salt_passage` rule, whose branches must be read in order
(`sharp_up` at +50 before `up` at +15), and the 35% of dev cases that sit within 5 pp of a
threshold, where the frozen policy already scores 0.456 against 0.547 elsewhere.

That is a much smaller and much better-posed question than the one `07` closed on, and it is
the natural `09`.

### For the README's comparison table

`07` said that table had an answer and that for the unprivileged case it was negative. That
stands, and it is now narrower rather than reversed. GRPO's group mean is the constant-per-
prompt baseline computed exactly; the learned critic here beats a constant by a real margin,
because dense per-field credit gives `V(s_t)` a within-sequence quantity to track that no
group mean can represent. **The per-token credit assignment is no longer vacuous.** It is
simply not what is limiting this task — and the ablation is the measurement that says so
rather than an argument that it should be.